# Improved Node-Edge Traversal Reconstruction

This notebook reconstructs a clean node-to-edge traversal sequence from raw tracking data, using graph structure to:
- Fill in missing nodes/edges based on connectivity
- Filter tracking noise and artifacts  
- Validate transitions follow graph logic
- Compute accurate durations at each location

## Requirements

- `summary_df.pkl` - Frame-by-frame tracking with `headstage_graph_node` and `headstage_graph_edge` columns
- Graph structure file (e.g., `FullMazeGrid.pkl`) with node positions and edge definitions

## Usage

1. Set the date variables (DD, MM, YYYY) to load your summary_df
2. Run all cells to reconstruct the improved traversal
3. Output is saved to `improved_traversal_df.pkl` and `.csv`

## Key Parameters

- `noise_threshold_frames=5` - Maximum frames for a location to be considered noise

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import pickle
import datetime
from pathlib import Path
import scipy.io


In [126]:
DD, MM, YYYY = "27", "08", "2025"
summary_pkl_path = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/summary_df.pkl"

summary_pkl_path

'/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/27_08_2025/summary_df.pkl'

In [127]:
summary_df = pd.read_pickle(summary_pkl_path)
summary_df.head()

headstage                               ear_L                         \
             x           y likelihood            x           y likelihood   
0    10.606056  933.431458   0.456236  1235.435181  183.376465   0.323545   
1  1116.935059  172.084854   0.120389  1126.294800  163.196243   0.054379   
2  1081.465088  841.832520   0.059083  1082.600952  636.113403   0.052841   
3  1075.816895  635.693420   0.058590  1082.294067  636.341675   0.062118   
4  1075.765991  635.866516   0.068334  1082.526489  636.290039   0.070915   

         ear_R                             midbody  ... frame_time_in_trial  \
             x           y likelihood            x  ...                       
0  1228.233032  190.214035   0.388586  1224.295044  ...               0.000   
1  1114.860840  183.646881   0.121726  1097.791016  ...               0.025   
2  1091.971680  838.912231   0.051133  1097.903809  ...               0.050   
3  1091.505005  839.112122   0.064682  1097.928711  ...               0.075   
4  1091.584961  839.085327   0.062680  1097.929077  ...               0.100   

  trial_length is_exact_multiple has_extra_frame headstage_graph_node  \
                                                                        
0      15.3437             False            True                  NaN   
1      15.3437             False            True                  NaN   
2      15.3437             False            True                  NaN   
3      15.3437             False            True                  NaN   
4      15.3437             False            True                  NaN   

  headstage_graph_edge fixed_node reward_node reward_size   lick  
                                                                  
0                 None        NaN       False         NaN  False  
1                 None        NaN       False         NaN  False  
2                 None        NaN       False         NaN  False  
3                 None        NaN       False         NaN  False  
4                 None        NaN       False         NaN  False  

[5 rows x 31 columns]

In [128]:
import networkx as nx
# Load the mapping file to get node positions
mapping_file = Path('/var/home/almogmeir/Documents/GitHub/NaviGraph/FullMazeGrid.pkl')
with open(mapping_file, 'rb') as f:
    mapping_data = pickle.load(f)

# Extract node and edge definitions
nodes_dict = mapping_data['mappings']['nodes']
edges_dict = mapping_data['mappings']['edges']

print(f"Found {len(nodes_dict)} nodes in mapping")
print(f"Found {len(edges_dict)} edge definitions")

# Convert node polygons to center positions
node_positions = {}
for node_id, polygon_list in nodes_dict.items():
    polygon = polygon_list[0] if polygon_list else []
    
    if polygon and len(polygon) >= 2:
        # Calculate center of bounding box
        xs = [p[0] for p in polygon]
        ys = [p[1] for p in polygon]
        center_x = sum(xs) / len(xs)
        center_y = sum(ys) / len(ys)
        node_positions[node_id] = (center_x, center_y)

print(f"Converted {len(node_positions)} node positions")

# Build graph structure from edges_dict
G = nx.DiGraph()

# Add all nodes
G.add_nodes_from(node_positions.keys())

# Add edges from edges_dict (bidirectional)
edges_added = 0
for edge_key, polygon_list in edges_dict.items():
    nodes = edge_key.split('_')
    if len(nodes) == 2:
        from_node, to_node = nodes
        if from_node in node_positions and to_node in node_positions:
            G.add_edge(from_node, to_node)
            G.add_edge(to_node, from_node)
            edges_added += 2

print(f"\nGraph loaded: {len(G.nodes)} nodes, {edges_added} edges (bidirectional)")
print(f"Graph ready for visualizations!")

Found 126 nodes in mapping
Found 125 edge definitions
Converted 126 node positions

Graph loaded: 126 nodes, 250 edges (bidirectional)
Graph ready for visualizations!


In [129]:
G.nodes

NodeView(('L34', 'L530', 'L52', 'L21', 'L30', 'R531', 'L56', 'R518', 'L517', 'R37', 'L510', 'R36', 'R530', 'L412', 'R525', 'R527', 'R57', 'L43', 'L41', 'L522', 'R45', 'R42', 'R512', 'L42', 'R46', 'R411', 'R32', 'R413', 'L525', 'R54', 'L527', 'R516', 'L529', 'R415', 'R517', 'R0', 'R23', 'R11', 'L58', 'R56', 'R514', 'R510', 'L48', 'L40', 'R48', 'L528', 'L526', 'R55', 'L513', 'L47', 'R43', 'R58', 'R21', 'R22', 'R50', 'L37', 'L36', 'R49', 'L515', 'L45', 'L51', 'L414', 'R414', 'L410', 'L23', 'L10', 'L514', 'R523', 'R526', 'L31', 'R33', 'L50', 'R519', 'L518', 'L22', 'L54', 'R515', 'R520', 'L0', 'L512', 'L49', 'L55', 'R10', 'R44', 'L516', 'R53', 'L531', 'R34', 'R511', 'R522', 'L519', 'L411', 'L523', 'L32', 'L53', 'L33', 'R524', 'R529', 'R528', 'L20', 'R20', 'R59', 'L511', 'L35', 'R40', 'R31', 'R410', 'L520', 'R51', 'R513', 'R412', 'L59', 'L11', 'L57', 'R47', 'L413', 'R521', 'R35', 'L524', 'L46', 'L44', 'L521', 'R41', 'L415', 'R52', 'R30'))

In [130]:
G.edges

OutEdgeView([('L34', 'L48'), ('L34', 'L22'), ('L34', 'L49'), ('L530', 'L415'), ('L52', 'L41'), ('L21', 'L10'), ('L21', 'L33'), ('L21', 'L32'), ('L30', 'L40'), ('L30', 'L20'), ('L30', 'L41'), ('R531', 'R415'), ('L56', 'L43'), ('R518', 'R49'), ('L517', 'L48'), ('R37', 'R23'), ('R37', 'R414'), ('R37', 'R415'), ('L510', 'L45'), ('R36', 'R412'), ('R36', 'R413'), ('R36', 'R23'), ('R530', 'R415'), ('L412', 'L36'), ('L412', 'L524'), ('L412', 'L525'), ('R525', 'R412'), ('R527', 'R413'), ('R57', 'R43'), ('L43', 'L57'), ('L43', 'L31'), ('L43', 'L56'), ('L41', 'L52'), ('L41', 'L30'), ('L41', 'L53'), ('L522', 'L411'), ('R45', 'R510'), ('R45', 'R511'), ('R45', 'R32'), ('R42', 'R55'), ('R42', 'R54'), ('R42', 'R31'), ('R512', 'R46'), ('L42', 'L31'), ('L42', 'L54'), ('L42', 'L55'), ('R46', 'R33'), ('R46', 'R513'), ('R46', 'R512'), ('R411', 'R522'), ('R411', 'R523'), ('R411', 'R35'), ('R32', 'R21'), ('R32', 'R44'), ('R32', 'R45'), ('R413', 'R526'), ('R413', 'R527'), ('R413', 'R36'), ('L525', 'L412'), ('

In [131]:
# Check summary_df structure
print("Summary DataFrame shape:", summary_df.shape)
print("\nColumns:", summary_df.columns.tolist())
print("\nFirst few rows of relevant columns:")
cols_to_check = ['headstage_graph_node', 'headstage_graph_edge', 'frame_time_in_trial']
if all(col in summary_df.columns for col in cols_to_check):
    print(summary_df[cols_to_check].head(20))

Summary DataFrame shape: (136968, 31)

Columns: [('headstage', 'x'), ('headstage', 'y'), ('headstage', 'likelihood'), ('ear_L', 'x'), ('ear_L', 'y'), ('ear_L', 'likelihood'), ('ear_R', 'x'), ('ear_R', 'y'), ('ear_R', 'likelihood'), ('midbody', 'x'), ('midbody', 'y'), ('midbody', 'likelihood'), ('tailbase', 'x'), ('tailbase', 'y'), ('tailbase', 'likelihood'), ('weighted', 'x'), ('weighted', 'y'), ('weighted', 'likelihood'), ('trial_idx', ''), ('frame_idx_global', ''), ('frame_idx_in_trial', ''), ('frame_time_in_trial', ''), ('trial_length', ''), ('is_exact_multiple', ''), ('has_extra_frame', ''), ('headstage_graph_node', ''), ('headstage_graph_edge', ''), ('fixed_node', ''), ('reward_node', ''), ('reward_size', ''), ('lick', '')]

First few rows of relevant columns:
   headstage_graph_node headstage_graph_edge frame_time_in_trial
                                                                
0                   NaN                 None               0.000
1                   NaN      

In [132]:
# Check where we have actual tracking data
node_col = ('headstage_graph_node', '')
edge_col = ('headstage_graph_edge', '')

print("Node tracking stats:")
print(f"  Total frames: {len(summary_df)}")
print(f"  Frames with node: {summary_df[node_col].notna().sum()}")
print(f"  Frames with edge: {summary_df[edge_col].notna().sum()}")
print(f"  Frames with both: {(summary_df[node_col].notna() & summary_df[edge_col].notna()).sum()}")
print(f"  Frames with neither: {(summary_df[node_col].isna() & summary_df[edge_col].isna()).sum()}")

# Show some rows with data
print("\nSample rows with node data:")
print(summary_df[summary_df[node_col].notna()].head(10)[[node_col, edge_col, ('frame_time_in_trial', '')]])

Node tracking stats:
  Total frames: 136968
  Frames with node: 75177
  Frames with edge: 56743
  Frames with both: 0
  Frames with neither: 5048

Sample rows with node data:
    headstage_graph_node headstage_graph_edge frame_time_in_trial
                                                                 
36                   L57                 None               0.900
40                   L50                 None               1.000
41                   L50                 None               1.025
88                    L0                 None               2.200
89                    L0                 None               2.225
90                    L0                 None               2.250
91                    L0                 None               2.275
92                    L0                 None               2.300
93                    L0                 None               2.325
445                   R0                 None              11.125


In [133]:
import pandas as pd
import numpy as np
import networkx as nx
import ast

class MazeTraversalFixer:
    def __init__(self, G, noise_threshold=5):
        self.G_orig = G
        self.noise_threshold = noise_threshold
        self.G_aug = self._build_augmented_graph(G)
        self.diagnostics = {
            'noise_smoothed': 0,
            'jumps_interpolated': 0,
            'nodes_edges_inferred': 0,
            'virtual_edges_ignored': 0,
            'invalid_unresolved': 0,
            'trials_processed': 0
        }

    def _is_virtual_edge(self, u, v):
        """Identifies Level 4 to Level 5 virtual edges to connect them directly."""
        u_str, v_str = str(u), str(v)
        def get_level(node_str):
            if len(node_str) >= 2 and node_str[0] in ('L', 'R'):
                return node_str[1]
            return None
        l_u, l_v = get_level(u_str), get_level(v_str)
        if l_u and l_v and {l_u, l_v} == {'4', '5'}:
            return True
        return False

    def _build_augmented_graph(self, G):
        G_aug = nx.Graph()
        for u in G.nodes():
            G_aug.add_node(u, state_type='node')
        for u, v in G.edges():
            if G.degree(u) == 1 or G.degree(v) == 1 or self._is_virtual_edge(u, v):
                G_aug.add_edge(u, v)
            else:
                edge_state = tuple(sorted((u, v)))
                G_aug.add_node(edge_state, state_type='edge')
                G_aug.add_edge(u, edge_state)
                G_aug.add_edge(v, edge_state)
        return G_aug

    def _is_missing(self, val):
        if val is None: return True
        if isinstance(val, (float, np.floating)) and np.isnan(val): return True
        if pd.api.types.is_scalar(val) and pd.isna(val): return True
        return False

    def _get_scalar(self, row, col_name):
        val = row.get(col_name)
        if isinstance(val, pd.Series):
            val = val.dropna()
            return val.iloc[0] if not val.empty else np.nan
        return val

    def _get_state(self, row):
        node = row.get('headstage_graph_node')
        if isinstance(node, pd.Series):
            node = node.dropna().iloc[0] if not node.dropna().empty else np.nan
        if not self._is_missing(node): return node
        
        edge = row.get('headstage_graph_edge')
        if isinstance(edge, pd.Series):
            edge = edge.dropna().iloc[0] if not edge.dropna().empty else np.nan
        if not self._is_missing(edge):
            if isinstance(edge, str):
                try: edge = ast.literal_eval(edge)
                except (ValueError, SyntaxError): pass
            if isinstance(edge, (list, tuple)) and len(edge) == 2:
                u, v = edge
                if self._is_virtual_edge(u, v):
                    self.diagnostics['virtual_edges_ignored'] += 1
                    return np.nan
                return tuple(sorted((u, v)))
            return edge
        return np.nan

    def _compress_to_runs(self, df_session):
        runs = []
        current_run = None
        
        for _, row in df_session.iterrows():
            state = self._get_state(row)
            f_global = self._get_scalar(row, 'frame_idx_global')
            f_trial = self._get_scalar(row, 'frame_idx_in_trial')
            t_idx = self._get_scalar(row, 'trial_idx')
            
            # Forward-fill NaNs ONLY if there is no gap in frame_idx_global
            if self._is_missing(state):
                if current_run is not None and (f_global - current_run['end_frame_global'] == 1):
                    current_run['end_frame_global'] = f_global
                    current_run['end_frame_in_trial'] = f_trial
                    current_run['duration'] += 1
                continue
                
            # Detect purposeful skipped frames (jumps in global time)
            is_gap = current_run is not None and (f_global - current_run['end_frame_global'] > 1)
            
            if (current_run is None or 
                current_run['state'] != state or 
                current_run['trial_idx'] != t_idx or
                is_gap):
                if current_run is not None:
                    runs.append(current_run)
                current_run = {
                    'state': state,
                    'start_frame_global': f_global,
                    'end_frame_global': f_global,
                    'start_frame_in_trial': f_trial,
                    'end_frame_in_trial': f_trial,
                    'trial_idx': t_idx,
                    'duration': 1
                }
            else:
                current_run['end_frame_global'] = f_global
                current_run['end_frame_in_trial'] = f_trial
                current_run['duration'] += 1
                
        if current_run is not None:
            runs.append(current_run)
        return runs

    def _fix_runs(self, runs):
        i = 0
        while i < len(runs) - 1:
            state1 = runs[i]['state']
            state2 = runs[i+1]['state']
            
            if state1 not in self.G_aug or state2 not in self.G_aug:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue

            if state1 == state2 or self.G_aug.has_edge(state1, state2):
                i += 1
                continue
                
            try:
                path = nx.shortest_path(self.G_aug, state1, state2)
            except nx.NetworkXNoPath:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue

            # Noise Smoothing: Change state instead of merging immediately, preserving gaps
            if runs[i+1]['duration'] <= self.noise_threshold:
                runs[i+1]['state'] = runs[i]['state']
                self.diagnostics['noise_smoothed'] += 1
                continue

            # Frame Stealing Logic
            missing_states = path[1:-1]
            num_missing = len(missing_states)
            
            # Figure out where to steal frames from (leave at least 1 frame in original runs)
            available_left = max(0, runs[i]['duration'] - 1)
            available_right = max(0, runs[i+1]['duration'] - 1)
            
            if available_left + available_right < num_missing:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue
                
            take_left = min(num_missing, available_left)
            take_right = num_missing - take_left
            
            orig_end_G = runs[i]['end_frame_global']
            orig_end_T = runs[i]['end_frame_in_trial']
            
            # Shrink Left Run
            if take_left > 0:
                runs[i]['end_frame_global'] -= take_left
                runs[i]['end_frame_in_trial'] -= take_left
                runs[i]['duration'] -= take_left
                
            orig_start_G = runs[i+1]['start_frame_global']
            orig_start_T = runs[i+1]['start_frame_in_trial']
            
            # Shrink Right Run
            if take_right > 0:
                runs[i+1]['start_frame_global'] += take_right
                runs[i+1]['start_frame_in_trial'] += take_right
                runs[i+1]['duration'] -= take_right
                
            # Inject Missing Nodes perfectly into the stolen real estate
            insert_idx = i + 1
            left_assigned = 0
            right_assigned = 0
            
            for state in missing_states:
                if left_assigned < take_left:
                    # Allocate frames leading up to the gap
                    frame_offset = take_left - left_assigned - 1
                    cur_G = orig_end_G - frame_offset
                    cur_T = orig_end_T - frame_offset
                    t_idx = runs[i]['trial_idx']
                    left_assigned += 1
                else:
                    # Allocate frames starting after the gap
                    cur_G = orig_start_G + right_assigned
                    cur_T = orig_start_T + right_assigned
                    t_idx = runs[i+1]['trial_idx']
                    right_assigned += 1
                    
                new_run = {
                    'state': state,
                    'start_frame_global': cur_G,
                    'end_frame_global': cur_G,
                    'start_frame_in_trial': cur_T,
                    'end_frame_in_trial': cur_T,
                    'trial_idx': t_idx,
                    'duration': 1
                }
                runs.insert(insert_idx, new_run)
                insert_idx += 1
                
            self.diagnostics['jumps_interpolated'] += 1
            self.diagnostics['nodes_edges_inferred'] += num_missing
            i = insert_idx

        return runs

    def validate(self, runs):
        for i in range(len(runs) - 1):
            if not self.G_aug.has_edge(runs[i]['state'], runs[i+1]['state']):
                return False, f"Invalid transition: index {i}: {runs[i]['state']} -> {runs[i+1]['state']}"
        return True, "Traversal valid."

    def process_session(self, summary_df):
        if 'trial_idx' in summary_df:
            t_col = summary_df['trial_idx']
            self.diagnostics['trials_processed'] = len(t_col.unique()) if not isinstance(t_col, pd.DataFrame) else len(t_col.iloc[:,0].unique())
            
        df_clean = summary_df.loc[:, ~summary_df.columns.duplicated()].copy()
        df_sorted = df_clean.sort_values('frame_idx_global')
        
        runs = self._compress_to_runs(df_sorted)
        fixed_runs = self._fix_runs(runs)
        
        # Final merge pass: Merge ONLY if there is NO gap in frame_idx_global
        final_runs = []
        for r in fixed_runs:
            if (final_runs and 
                final_runs[-1]['state'] == r['state'] and 
                final_runs[-1]['trial_idx'] == r['trial_idx'] and
                r['start_frame_global'] - final_runs[-1]['end_frame_global'] <= 1):
                
                final_runs[-1]['end_frame_global'] = r['end_frame_global']
                final_runs[-1]['end_frame_in_trial'] = r['end_frame_in_trial']
                final_runs[-1]['duration'] += r['duration']
            else:
                final_runs.append(r)
        
        is_valid, msg = self.validate(final_runs)
        if not is_valid: print(f"Warning: {msg}")
            
        traversal_df = pd.DataFrame(final_runs)
        
        traversal_df['location_type'] = traversal_df['state'].apply(
            lambda x: 'node' if isinstance(x, (int, str)) and not isinstance(x, tuple) else 'edge'
        )
        traversal_df['headstage_graph_node'] = traversal_df.apply(
            lambda row: row['state'] if row['location_type'] == 'node' else np.nan, axis=1
        )
        traversal_df['headstage_graph_edge'] = traversal_df.apply(
            lambda row: row['state'] if row['location_type'] == 'edge' else np.nan, axis=1
        )
        
        cols = ['headstage_graph_node', 'headstage_graph_edge', 'location_type',
                'start_frame_global', 'end_frame_global', 'start_frame_in_trial', 
                'end_frame_in_trial', 'trial_idx', 'duration']
        
        final_df = traversal_df[cols]
        numeric_cols = ['start_frame_global', 'end_frame_global', 'start_frame_in_trial', 'end_frame_in_trial', 'duration']
        for col in numeric_cols:
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
            
        return final_df
        
    def print_diagnostics(self):
        print("\n--- Traversal Diagnostics ---")
        print(f"Trials Detected: {self.diagnostics['trials_processed']}")
        print(f"L4/L5 Virtual Edges Reassigned: {self.diagnostics['virtual_edges_ignored']} frames")
        print(f"Short Noise Jumps Smoothed: {self.diagnostics['noise_smoothed']}")
        print(f"Valid Jumps Interpolated: {self.diagnostics['jumps_interpolated']}")
        print(f"Total Nodes/Edges Inferred: {self.diagnostics['nodes_edges_inferred']}")
        if self.diagnostics['invalid_unresolved'] > 0:
            print(f"⚠️ Unresolved Tracking Errors: {self.diagnostics['invalid_unresolved']}")
        print("-----------------------------\n")

In [134]:
import pandas as pd
import numpy as np
import networkx as nx
import ast

class MazeTraversalFixer:
    def __init__(self, G, noise_threshold=5):
        self.noise_threshold = noise_threshold
        # Build graph immediately with strict string formatting
        self.G_aug = self._build_augmented_graph(G)
        self.diagnostics = {
            'noise_smoothed': 0,
            'jumps_interpolated': 0,
            'nodes_edges_inferred': 0,
            'virtual_edges_ignored': 0,
            'invalid_unresolved': 0,
            'trials_processed': 0
        }

    def _is_virtual_edge(self, u_str, v_str):
        """Identifies Level 4 to Level 5 virtual edges to connect them directly."""
        def get_level(node_str):
            if len(node_str) >= 2 and node_str[0] in ('L', 'R'):
                return node_str[1]
            return None
        l_u, l_v = get_level(u_str), get_level(v_str)
        if l_u and l_v and {l_u, l_v} == {'4', '5'}:
            return True
        return False

    def _build_augmented_graph(self, G):
        """Transforms spatial graph into state-transition graph, strictly enforcing strings."""
        G_aug = nx.Graph()
        
        # Force all core nodes to be pure strings
        for u in G.nodes():
            G_aug.add_node(str(u), state_type='node')
            
        for u, v in G.edges():
            u_str, v_str = str(u), str(v)
            
            if G.degree(u) == 1 or G.degree(v) == 1 or self._is_virtual_edge(u_str, v_str):
                G_aug.add_edge(u_str, v_str)
            else:
                # Force strictly sorted tuple of strings (e.g., ('L34', 'L49'))
                edge_state = tuple(sorted((u_str, v_str)))
                G_aug.add_node(edge_state, state_type='edge')
                G_aug.add_edge(u_str, edge_state)
                G_aug.add_edge(v_str, edge_state)
                
        return G_aug

    def _is_missing(self, val):
        if val is None: return True
        if isinstance(val, (float, np.floating)) and np.isnan(val): return True
        if pd.api.types.is_scalar(val) and pd.isna(val): return True
        return False

    def _get_scalar(self, row, col_name):
        val = row.get(col_name)
        if isinstance(val, pd.Series):
            val = val.dropna()
            return val.iloc[0] if not val.empty else np.nan
        return val

    def _get_state(self, row):
        """Extracts and heavily standardizes the state to match G_aug perfectly."""
        # 1. Handle Node
        node = row.get('headstage_graph_node')
        if isinstance(node, pd.Series):
            node = node.dropna().iloc[0] if not node.dropna().empty else np.nan
            
        if not self._is_missing(node): 
            # If it imported as a float like 34.0, strip the decimal
            if isinstance(node, float) and node.is_integer():
                return str(int(node))
            return str(node)
        
        # 2. Handle Edge
        edge = row.get('headstage_graph_edge')
        if isinstance(edge, pd.Series):
            edge = edge.dropna().iloc[0] if not edge.dropna().empty else np.nan
            
        if not self._is_missing(edge):
            if isinstance(edge, str):
                try: 
                    edge = ast.literal_eval(edge)
                except (ValueError, SyntaxError): 
                    # Fallback: if string is literally "(L49, L34)" without internal quotes
                    clean_str = edge.strip("() ")
                    if ',' in clean_str:
                        parts = [p.strip(" '\"") for p in clean_str.split(',')]
                        if len(parts) == 2:
                            edge = parts
                            
            if isinstance(edge, (list, tuple)) and len(edge) == 2:
                # Strictly cast to string and sort
                u_str, v_str = str(edge[0]), str(edge[1])
                if self._is_virtual_edge(u_str, v_str):
                    self.diagnostics['virtual_edges_ignored'] += 1
                    return np.nan
                return tuple(sorted((u_str, v_str)))
                
        return np.nan

    def _compress_to_runs(self, df_session):
        runs = []
        current_run = None
        
        for _, row in df_session.iterrows():
            state = self._get_state(row)
            f_global = self._get_scalar(row, 'frame_idx_global')
            f_trial = self._get_scalar(row, 'frame_idx_in_trial')
            t_idx = self._get_scalar(row, 'trial_idx')
            
            # Forward-fill NaNs ONLY if there is no gap in frame_idx_global
            if self._is_missing(state):
                if current_run is not None and (f_global - current_run['end_frame_global'] == 1):
                    current_run['end_frame_global'] = f_global
                    current_run['end_frame_in_trial'] = f_trial
                    current_run['duration'] += 1
                continue
                
            is_gap = current_run is not None and (f_global - current_run['end_frame_global'] > 1)
            
            if (current_run is None or 
                current_run['state'] != state or 
                current_run['trial_idx'] != t_idx or
                is_gap):
                if current_run is not None:
                    runs.append(current_run)
                current_run = {
                    'state': state,
                    'start_frame_global': f_global,
                    'end_frame_global': f_global,
                    'start_frame_in_trial': f_trial,
                    'end_frame_in_trial': f_trial,
                    'trial_idx': t_idx,
                    'duration': 1
                }
            else:
                current_run['end_frame_global'] = f_global
                current_run['end_frame_in_trial'] = f_trial
                current_run['duration'] += 1
                
        if current_run is not None:
            runs.append(current_run)
        return runs

    def _fix_runs(self, runs):
        i = 0
        while i < len(runs) - 1:
            state1 = runs[i]['state']
            state2 = runs[i+1]['state']
            
            if state1 not in self.G_aug or state2 not in self.G_aug:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue

            if state1 == state2 or self.G_aug.has_edge(state1, state2):
                i += 1
                continue
                
            try:
                path = nx.shortest_path(self.G_aug, state1, state2)
            except nx.NetworkXNoPath:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue

            if runs[i+1]['duration'] <= self.noise_threshold:
                runs[i+1]['state'] = runs[i]['state']
                self.diagnostics['noise_smoothed'] += 1
                continue

            missing_states = path[1:-1]
            num_missing = len(missing_states)
            
            available_left = max(0, runs[i]['duration'] - 1)
            available_right = max(0, runs[i+1]['duration'] - 1)
            
            if available_left + available_right < num_missing:
                self.diagnostics['invalid_unresolved'] += 1
                i += 1
                continue
                
            take_left = min(num_missing, available_left)
            take_right = num_missing - take_left
            
            orig_end_G = runs[i]['end_frame_global']
            orig_end_T = runs[i]['end_frame_in_trial']
            
            if take_left > 0:
                runs[i]['end_frame_global'] -= take_left
                runs[i]['end_frame_in_trial'] -= take_left
                runs[i]['duration'] -= take_left
                
            orig_start_G = runs[i+1]['start_frame_global']
            orig_start_T = runs[i+1]['start_frame_in_trial']
            
            if take_right > 0:
                runs[i+1]['start_frame_global'] += take_right
                runs[i+1]['start_frame_in_trial'] += take_right
                runs[i+1]['duration'] -= take_right
                
            insert_idx = i + 1
            left_assigned = 0
            right_assigned = 0
            
            for state in missing_states:
                if left_assigned < take_left:
                    frame_offset = take_left - left_assigned - 1
                    cur_G = orig_end_G - frame_offset
                    cur_T = orig_end_T - frame_offset
                    t_idx = runs[i]['trial_idx']
                    left_assigned += 1
                else:
                    cur_G = orig_start_G + right_assigned
                    cur_T = orig_start_T + right_assigned
                    t_idx = runs[i+1]['trial_idx']
                    right_assigned += 1
                    
                new_run = {
                    'state': state,  # This state is pulled straight from G_aug, so it's already stringified/sorted perfectly
                    'start_frame_global': cur_G,
                    'end_frame_global': cur_G,
                    'start_frame_in_trial': cur_T,
                    'end_frame_in_trial': cur_T,
                    'trial_idx': t_idx,
                    'duration': 1
                }
                runs.insert(insert_idx, new_run)
                insert_idx += 1
                
            self.diagnostics['jumps_interpolated'] += 1
            self.diagnostics['nodes_edges_inferred'] += num_missing
            i = insert_idx

        return runs

    def validate(self, runs):
        for i in range(len(runs) - 1):
            if not self.G_aug.has_edge(runs[i]['state'], runs[i+1]['state']):
                return False, f"Invalid transition: index {i}: {runs[i]['state']} -> {runs[i+1]['state']}"
        return True, "Traversal valid."

    def process_session(self, summary_df):
        if 'trial_idx' in summary_df:
            t_col = summary_df['trial_idx']
            self.diagnostics['trials_processed'] = len(t_col.unique()) if not isinstance(t_col, pd.DataFrame) else len(t_col.iloc[:,0].unique())
            
        df_clean = summary_df.loc[:, ~summary_df.columns.duplicated()].copy()
        df_sorted = df_clean.sort_values('frame_idx_global')
        
        runs = self._compress_to_runs(df_sorted)
        fixed_runs = self._fix_runs(runs)
        
        final_runs = []
        for r in fixed_runs:
            if (final_runs and 
                final_runs[-1]['state'] == r['state'] and 
                final_runs[-1]['trial_idx'] == r['trial_idx'] and
                r['start_frame_global'] - final_runs[-1]['end_frame_global'] <= 1):
                
                final_runs[-1]['end_frame_global'] = r['end_frame_global']
                final_runs[-1]['end_frame_in_trial'] = r['end_frame_in_trial']
                final_runs[-1]['duration'] += r['duration']
            else:
                final_runs.append(r)
        
        is_valid, msg = self.validate(final_runs)
        if not is_valid: print(f"Warning: {msg}")
            
        traversal_df = pd.DataFrame(final_runs)
        
        # State assignment is strictly divided by tuple vs string
        traversal_df['location_type'] = traversal_df['state'].apply(
            lambda x: 'edge' if isinstance(x, tuple) else 'node'
        )
        traversal_df['headstage_graph_node'] = traversal_df.apply(
            lambda row: row['state'] if row['location_type'] == 'node' else np.nan, axis=1
        )
        traversal_df['headstage_graph_edge'] = traversal_df.apply(
            lambda row: row['state'] if row['location_type'] == 'edge' else np.nan, axis=1
        )
        
        cols = ['headstage_graph_node', 'headstage_graph_edge', 'location_type',
                'start_frame_global', 'end_frame_global', 'start_frame_in_trial', 
                'end_frame_in_trial', 'trial_idx', 'duration']
        
        final_df = traversal_df[cols]
        numeric_cols = ['start_frame_global', 'end_frame_global', 'start_frame_in_trial', 'end_frame_in_trial', 'duration']
        for col in numeric_cols:
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
            
        return final_df
        
    def print_diagnostics(self):
        print("\n--- Traversal Diagnostics ---")
        print(f"Trials Detected: {self.diagnostics['trials_processed']}")
        print(f"L4/L5 Virtual Edges Reassigned: {self.diagnostics['virtual_edges_ignored']} frames")
        print(f"Short Noise Jumps Smoothed: {self.diagnostics['noise_smoothed']}")
        print(f"Valid Jumps Interpolated: {self.diagnostics['jumps_interpolated']}")
        print(f"Total Nodes/Edges Inferred: {self.diagnostics['nodes_edges_inferred']}")
        if self.diagnostics['invalid_unresolved'] > 0:
            print(f"⚠️ Unresolved Tracking Errors: {self.diagnostics['invalid_unresolved']}")
        print("-----------------------------\n")

In [135]:
# 1. Initialize the fixer with your Maze Graph
# noise_threshold dictates how many frames (duration) a jump must be to be considered real vs noise
fixer = MazeTraversalFixer(G, noise_threshold=5)

# 2. Process your session dataframe
traversal_df = fixer.process_session(summary_df)

# 3. Review the diagnostics
fixer.print_diagnostics()

# 4. Preview the resulting node-edge-node traversal
display(traversal_df.head(15))


--- Traversal Diagnostics ---
Trials Detected: 541
L4/L5 Virtual Edges Reassigned: 0 frames
Short Noise Jumps Smoothed: 703
Valid Jumps Interpolated: 640
Total Nodes/Edges Inferred: 1464
-----------------------------



,headstage_graph_node,headstage_graph_edge,location_type,start_frame_global,end_frame_global,start_frame_in_trial,end_frame_in_trial,trial_idx,duration
0,NaN,"(L31, L43)",edge,32,40,32,40,0,9
1,L31,NaN,node,41,41,41,41,0,1
2,NaN,"(L20, L31)",edge,42,42,42,42,0,1
3,L20,NaN,node,43,43,43,43,0,1
4,NaN,"(L10, L20)",edge,44,44,44,44,0,1
5,L10,NaN,node,45,45,45,45,0,1
6,NaN,"(L0, L10)",edge,46,87,46,87,0,42
7,L0,NaN,node,88,93,88,93,0,6
8,NaN,"(L0, R0)",edge,94,444,94,444,0,351
9,R0,NaN,node,445,445,445,445,0,1


In [136]:
traversal_df.to_csv(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/traversal_df.csv", index=False)

# Work with the new traversal_df for better analysis